# Panel de sucursales por provincia

Apila los archivos `Distrib/Suc_pcia/{codigo}.txt` de los 133 dumps IEF. Cada archivo es un snapshot a la fecha del dump con la cantidad de sucursales (plenas, móviles, cajeros, terminales, etc.) por entidad × provincia.

Output: `data/interim/paneles_largos/panel_sucursales_provincia.parquet` con una observación por `(codigo_entidad, dump_yyyymm, codigo_provincia)`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

IEF_ROOT = RAW / "bcra_ief"
OUT = PANELES / "panel_sucursales_provincia.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

## Apilado

In [ ]:
dumps = sorted([d for d in IEF_ROOT.iterdir() if d.is_dir() and d.name.isdigit()])
cols = ["codigo_entidad", "nombre_entidad", "fecha_corte", "codigo_provincia", "nombre_provincia",
        "sucursales_plenas", "sucursales_op_especifica", "sucursales_moviles",
        "dependencias_automatizadas", "cajeros_automaticos", "terminales_autoservicio",
        "puestos_promocion", "agencias_complementarias", "cantidad_entidades"]

bloques = []
for d in dumps:
    yyyymm_dump = d.name
    sc_dir = d / "Entfin/Distrib/Suc_pcia"
    if not sc_dir.exists():
        continue
    for f in sc_dir.glob("*.txt"):
        if f.name == "formato.txt":
            continue
        try:
            df = pd.read_csv(f, sep="\t", header=None, names=cols, encoding="ISO-8859-1",
                             dtype=str)  # leo TODO como str para evitar inferencias inconsistentes
            df["dump_yyyymm"] = int(yyyymm_dump)
            bloques.append(df)
        except Exception:
            pass

raw = pd.concat(bloques, ignore_index=True)
# Conversion explícita de columnas numéricas
for c in ["sucursales_plenas", "sucursales_op_especifica", "sucursales_moviles",
          "dependencias_automatizadas", "cajeros_automaticos", "terminales_autoservicio",
          "puestos_promocion", "agencias_complementarias", "cantidad_entidades"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
print(f"Filas apiladas: {len(raw):,}")

## Deduplicación

Para cada `(codigo_entidad, fecha_corte, codigo_provincia)` me quedo con la observación del dump más reciente. (Si dos dumps reportan la misma fecha_corte, asumo que el más reciente es el más correcto.)

In [ ]:
raw["fecha_corte_int"] = raw["fecha_corte"].astype(int)
raw = raw.sort_values(["codigo_entidad", "fecha_corte_int", "codigo_provincia", "dump_yyyymm"])
panel = raw.drop_duplicates(subset=["codigo_entidad", "fecha_corte_int", "codigo_provincia"], keep="last")
panel = panel.drop(columns=["fecha_corte_int"])
print(f"Filas tras deduplicar: {len(panel):,}")

## Persistencia

In [ ]:
panel.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(panel):,}")

## Validaciones

In [ ]:
assert panel[["codigo_entidad", "fecha_corte", "codigo_provincia"]].duplicated().sum() == 0
print("Validaciones OK")
print(f"  entidades: {panel['codigo_entidad'].nunique()}")
print(f"  provincias: {panel['codigo_provincia'].nunique()}")
print(f"  cortes: {panel['fecha_corte'].nunique()}")

## Muestra: red de sucursales del Banco Nación al último corte

In [ ]:
duckdb.sql(f"""
    select nombre_provincia, sucursales_plenas, cajeros_automaticos
    from '{OUT}'
    where codigo_entidad = '00011' and fecha_corte = (select max(fecha_corte) from '{OUT}')
    order by sucursales_plenas desc
""").df()